In [1]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from gensim.models import Word2Vec
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter
from tqdm import tqdm
import itertools

In [5]:
# Setar seed para reprodutibilidade
torch.manual_seed(42)
np.random.seed(42)

# Configurar dispositivo (GPU se disponível, senão CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizando dispositivo: {device}")

Utilizando dispositivo: cpu


In [6]:
# 1. CARREGAMENTO DOS DADOS
# -----------------------------------------------------------------------------
DATA_PATH = "train.xlsx"
TEXT_COL = "resp_text"      # Mude para o nome da coluna de texto
TARGET_COL = "clarity"   # Mude para o nome da coluna de rótulos

df = pd.read_excel(DATA_PATH)
# df.describe()

label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {v: k for k, v in label2id.items()}
df["label"] = df[TARGET_COL].map(label2id)

In [7]:
# 1. NORMALIZAÇÃO DE TEXTO (REGEX)
# ==========================================
def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    # 1. URLs -> LINK_URL
    texto = re.sub(r'https?://\S+|www\.\S+', 'LINK_URL', texto)
    # 2. Leis e Decretos -> LEI_DECRETO
    texto = re.sub(r'(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*', 'LEI_DECRETO', texto)
    # 3. Números de Processo -> NUMERO_PROCESSO
    texto = re.sub(r'\b\d{5,}\.\d{6}/\d{4}-\d{2}\b', 'NUMERO_PROCESSO', texto)
    # 4. Telefone -> NUMERO_TELEFONE
    texto = re.sub(r'\(?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}', 'NUMERO_TELEFONE', texto)
    return texto.lower()

def tokenize(texto):
    texto = normalizar_texto(texto)
    return texto.split()

In [8]:
# 2. WORD2VEC & CONSTRUÇÃO DE VOCABULÁRIO (CORRIGIDA)
# ==========================================
def build_vocab_and_word2vec(sentences, embedding_dim=100):
    # Treinar Word2Vec
    w2v_model = Word2Vec(
        sentences=sentences,
        vector_size=embedding_dim,
        window=5,
        min_count=2,
        workers=4,
        seed=42
    )
    
    # Criar mapeamento palavra -> id (0 é reservado para <PAD>)
    vocab = {"<PAD>": 0, "<UNK>": 1}
    weights = [np.zeros(embedding_dim), np.random.normal(scale=0.6, size=(embedding_dim,))]
    
    for word in w2v_model.wv.index_to_key:
        vocab[word] = len(vocab)
        weights.append(w2v_model.wv[word])
        
    weights_matrix = torch.tensor(np.array(weights), dtype=torch.float32)
    
    # IMPORTANTE: Retorno explícito para evitar o TypeError
    return vocab, weights_matrix

In [13]:
# 3. DATASET E MODELO GRU COM EMBEDDINGS
# ==========================================
class TextDataset(Dataset):
    def __init__(self, texts_tokens, labels, vocab):
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.sequences = []
        
        for tokens in texts_tokens:
            seq = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
            if len(seq) == 0:
                seq = [vocab["<PAD>"]]
            self.sequences.append(torch.tensor(seq, dtype=torch.long))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

def collate_fn(batch):
    sequences, labels = zip(*batch)
    padded_seqs = pad_sequence(sequences, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return padded_seqs, labels

class GRUClassifier(nn.Module):
    def __init__(self, weights_matrix, hidden_dim, num_classes):
        super(GRUClassifier, self).__init__()
        # Inicializa com a matriz pré-treinada do Word2Vec
        self.embedding = nn.Embedding.from_pretrained(weights_matrix, freeze=False, padding_idx=0)
        
        # BiGRU (Bidirecional aprende contexto da esquerda p/ direita e vice-versa)
        self.gru = nn.GRU(weights_matrix.shape[1], hidden_dim, batch_first=True, bidirectional=True)
        
        # Como é Bidirecional, a saída da GRU tem dimensão hidden_dim * 2
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)  # [batch_size, seq_len, embed_dim]
        out, _ = self.gru(embedded)   # [batch_size, seq_len, hidden_dim * 2]
        
        # Criar máscara para ignorar os tokens de padding (<PAD> = 0)
        mask = (x != 0).unsqueeze(-1).float() # [batch_size, seq_len, 1]
        
        # Multiplica a saída da GRU pela máscara para zerar as posições de padding
        out_masked = out * mask
        
        # Média apenas sobre os tokens válidos (Mean Pooling)
        sum_out = out_masked.sum(dim=1)
        clamp_len = mask.sum(dim=1).clamp(min=1) # Evita divisão por zero
        mean_pooled = sum_out / clamp_len
        
        logits = self.fc(mean_pooled)
        return logits

In [15]:
# 4. LOOP DE GRID SEARCH COM 5-FOLD CV
# ==========================================

X_raw_texts = df.resp_text
y_labels = df.label

# 4. GRID SEARCH ENXUTO PARA CPU (4 COMBINAÇÕES)
# ==========================================
param_grid = {
    'embedding_dim': [100],         # 100 dimensões é um padrão robusto
    'hidden_dim': [64, 128],        # Compara uma GRU leve x uma maior
    'lr': [0.001],                  # Taxa de aprendizagem padrão
    'batch_size': [64, 128]         # Batches maiores aceleram o processamento na CPU
}

keys, values = zip(*param_grid.items())
all_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

# Pré-tokenização de todo o dataset
X_tokens_all = [tokenize(t) for t in X_raw_texts]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
device = torch.device('cpu') # Treino forçado em CPU

best_f1_global = 0.0
best_config = None

print(f"A executar Grid Search na CPU ({len(all_combinations)} combinações no total)...")

for config_idx, params in enumerate(all_combinations):
    print(f"\n==================================================")
    print(f"Configuração {config_idx + 1}/{len(all_combinations)}: {params}")
    print(f"==================================================")
    
    fold_f1_scores = []
    fold_acc_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_tokens_all, y_labels)):
        X_train_tokens = [X_tokens_all[i] for i in train_idx]
        y_train = [y_labels[i] for i in train_idx]
        
        X_val_tokens = [X_tokens_all[i] for i in val_idx]
        y_val = [y_labels[i] for i in val_idx]
        
        # 1. Word2Vec sem data-leakage
        vocab, weights_matrix = build_vocab_and_word2vec(
            X_train_tokens, embedding_dim=params['embedding_dim']
        )
        
        # 2. DataLoaders
        train_ds = TextDataset(X_train_tokens, y_train, vocab)
        val_ds = TextDataset(X_val_tokens, y_val, vocab)
        
        train_loader = DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_ds, batch_size=params['batch_size'], shuffle=False, collate_fn=collate_fn)
        
        # 3. Modelo
        num_classes = len(set(y_labels))
        model = GRUClassifier(weights_matrix, params['hidden_dim'], num_classes).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'])
        
        # 4. Treino curto (3 Épocas)
        for epoch in range(8):
            model.train()
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                
        # 5. Avaliação
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                outputs = model(inputs)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_targets.extend(labels.numpy())
                
        acc = accuracy_score(val_targets, val_preds)
        f1 = f1_score(val_targets, val_preds, average='macro')
        
        fold_acc_scores.append(acc)
        fold_f1_scores.append(f1)
        print(f"  Fold {fold + 1} -> Acurácia: {acc:.4f} | F1-Macro: {f1:.4f}")
        
    mean_acc = np.mean(fold_acc_scores)
    mean_f1 = np.mean(fold_f1_scores)
    
    print(f"--> Média Config {config_idx + 1} -> Acurácia: {mean_acc:.4f} | F1-Macro: {mean_f1:.4f}")
    
    if mean_f1 > best_f1_global:
        best_f1_global = mean_f1
        best_config = params

print("\n==========================================")
print("RESULTADO FINAL DO GRID SEARCH")
print(f"Melhor Configuração: {best_config}")
print(f"Melhor F1-Macro Médio: {best_f1_global:.4f}")
print("==========================================")

A executar Grid Search na CPU (4 combinações no total)...

Configuração 1/4: {'embedding_dim': 100, 'hidden_dim': 64, 'lr': 0.001, 'batch_size': 64}


KeyboardInterrupt: 

In [3]:
import torch
print("CUDA disponível?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nome da GPU:", torch.cuda.get_device_name(0))

CUDA disponível?: False
